# 02 · Feature Engineering
Three representations are compared under identical protocol:

| name | definition | dim |
|---|---|---|
| `pixels` | flattened intensities in [0,1] | 784 |
| `hog` | Histogram of Oriented Gradients (skimage) | 864 |
| `hog_stats` | HOG + 16 hand-crafted shape statistics | 880 |

In [ ]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src.config import *
from src.data_loader import load_raw, to_unit, make_splits
from src.plotting import set_style
set_style()
def show(name, w=900): display(Image(filename=str(FIG_DIR / f'{name}.png'), width=w))
from src.features import *
S = make_splits()

## 1. Orientation check and scaling
Pixels are scaled to [0,1], labels untouched. Images are stored upright, so `skimage.feature.hog` is applied on the (row, col) array directly.

## 2. How HOG works
1. Compute horizontal/vertical gradients `gx, gy`.
2. Magnitude `sqrt(gx²+gy²)` and orientation `atan2(gy, gx)` per pixel.
3. Split the image into cells (4×4 px); each cell makes a histogram of orientations, weighted by magnitude.
4. Group 2×2 cells into overlapping blocks and L2-Hys normalise (illumination invariance).
5. Concatenate all block vectors.

In [ ]:
show('fe_02_hog_visual', 1000)

## 3. HOG hyper-parameter ablation
Grid: orientations {6,9,12} × cell {2,4,7} × block {1,2}; scored on validation with a Logistic-Regression probe (12k training images).

In [ ]:
pd.read_csv(TABLE_DIR/'fe_hog_ablation.csv').head(12)

In [ ]:
show('fe_01_hog_ablation', 1000)

**Decision:** finer cells improve accuracy but explode dimensionality (4k-8k) and slow kNN/SVM/RF. We choose the best configuration under a 2,000-d budget: **6 orientations, 4×4 cells, 2×2 blocks (864-d)**.

## 4. Hand-crafted shape statistics
16 interpretable global descriptors: intensity mean/std, ink fraction, bounding-box height/width/aspect, centroid, left-right and top-bottom symmetry, Sobel edge density, top/bottom mass, width of top/bottom rows and the gap between trouser legs.

In [ ]:
show('fe_03_shape_stats_profile', 900); show('fe_04_shape_stats_importance', 500)

## 5. PCA on pixels (reference compression)

In [ ]:
show('fe_05_pca_sweep', 520); pd.read_csv(TABLE_DIR/'fe_pca_sweep.csv')

## 6. Feature-set comparison (same LR probe)

In [ ]:
show('fe_06_feature_set_probe', 520); pd.read_csv(TABLE_DIR/'fe_feature_set_probe.csv')

The probe is deliberately simple; the full 4-model comparison is in notebook 03.